In [ ]:
# ==============================================================================
# Direct Multimodal LLM Preference Analysis & Evaluation Pipeline
# Per-Annotator Version: reads from ./annotator/[ID].csv, images from ./screen/
# Outputs to ./results/[ID].json
# ==============================================================================
import os
import json
import base64
import time
import random
import re
import urllib.request
import urllib.error
import http.client
from typing import List, Dict, Any, Optional
from pathlib import Path
import pandas as pd
from PIL import Image

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

# ==============================================================================
# CONFIGURATION
# ==============================================================================
ANNOTATOR_DIR = "./annotator"
IMAGES_DIR = "./screen"
RESULTS_DIR = "./results"

MODEL = "gpt-5.4-mini"
SAMPLE_SIZE = 40       # Total rows to sample per annotator
TRAIN_SIZE = 20        # Used for preference learning (Phase A + C)
EVAL_SIZE = 20         # Used for validation (Phase D)
SEED = 7

AZURE_OPENAI_TARGET_URI = os.getenv(
    "AZURE_OPENAI_TARGET_URI",
    f"https://wu-lab-east-us-2.openai.azure.com/openai/deployments/{MODEL}/chat/completions?api-version=2025-01-01-preview"
)
AZURE_OPENAI_API_KEY="XXX"

# ==============================================================================
# HELPER FUNCTIONS
# ==============================================================================
def set_seed(seed: int):
    random.seed(seed)

MAX_SIZE = (1024, 1024)

def encode_image(image_path: str) -> Optional[str]:
    try:
        img = Image.open(image_path).convert("RGB")
        img.thumbnail(MAX_SIZE, Image.Resampling.LANCZOS)
        import io
        buf = io.BytesIO()
        img.save(buf, format="JPEG", quality=85)
        return base64.b64encode(buf.getvalue()).decode("utf-8").replace("\n", "")
    except Exception as e:
        print(f"  [!] Error encoding image {image_path}: {e}")
        return None

def extract_json_from_text(text: str) -> dict:
    try:
        match = re.search(r'```(?:json)?\s*(.*?)\s*```', text, re.DOTALL)
        if match:
            return json.loads(match.group(1))
        return json.loads(text)
    except json.JSONDecodeError as e:
        print(f"  [!] JSON Parse Error. Raw text:\n{text}\n")
        raise e

def is_valid_phase_a_result(result: Any) -> bool:
    if not isinstance(result, dict):
        return False
    preferred_features = result.get("preferred_features")
    return isinstance(preferred_features, list)

def is_valid_phase_d_result(result: Any) -> bool:
    if not isinstance(result, dict):
        return False
    criteria_evaluations = result.get("criteria_evaluations")
    return isinstance(criteria_evaluations, list)

def call_llm(
    prompt: str,
    image_a_b64: Optional[str] = None,
    image_b_b64: Optional[str] = None,
    max_retries: int = 5
) -> Dict[str, Any]:
    content = [{"type": "text", "text": prompt}]

    if image_a_b64 and image_b_b64:
        content.extend([
            {"type": "text", "text": "Image A:"},
            {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{image_a_b64}"}},
            {"type": "text", "text": "Image B:"},
            {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{image_b_b64}"}},
        ])

    # Azure OpenAI: model is specified in the deployment URL, not the payload.
    # Including it can cause a 400 if the value doesn't match the deployment name.
    payload = {
        "messages": [{"role": "user", "content": content}],
        "temperature": 0.2,
        "max_completion_tokens": 1000,
    }
    headers = {
        "Content-Type": "application/json",
        "api-key": AZURE_OPENAI_API_KEY,
        "User-Agent": "Jupyter-LLM-Evaluator/1.0",
    }
    data = json.dumps(payload).encode("utf-8")

    for attempt in range(max_retries):
        try:
            req = urllib.request.Request(
                AZURE_OPENAI_TARGET_URI, data=data, headers=headers, method="POST"
            )
            with urllib.request.urlopen(req, timeout=120) as response:
                result = json.loads(response.read().decode("utf-8"))
                content_str = result["choices"][0]["message"]["content"]
                return extract_json_from_text(content_str)
        except urllib.error.HTTPError as e:
            # Read Azure's full error body so the real reason is visible
            try:
                body = e.read().decode("utf-8")
                print(f"  [!] HTTP {e.code} (Attempt {attempt+1}/{max_retries}): {body}")
            except Exception:
                print(f"  [!] HTTP {e.code} (Attempt {attempt+1}/{max_retries}): {e.reason}")
            if e.code == 400:
                raise  # Bad request won't improve with retries
            if attempt < max_retries - 1:
                time.sleep(2 + (2 ** attempt))
            else:
                raise
        except (urllib.error.URLError, http.client.RemoteDisconnected, ConnectionResetError) as e:
            print(f"  [!] Connection Error (Attempt {attempt+1}/{max_retries}): {e}")
            if attempt < max_retries - 1:
                time.sleep(2 + (2 ** attempt))
            else:
                raise
        except Exception as e:
            print(f"  [!] API Error (Attempt {attempt+1}/{max_retries}): {e}")
            if attempt < max_retries - 1:
                time.sleep(2 + (2 ** attempt))
            else:
                raise
    return {}


# ==============================================================================
# DATA HELPERS
# ==============================================================================
def get_winner_label(row: pd.Series) -> str:
    """
    Map final_choice strings to 'A' or 'B'.
    'A > B'  / 'A >> B'  → user preferred A
    'A < B'  / 'A << B'  → user preferred B
    """
    choice = str(row.get("final_choice", "")).strip()
    if choice in ("A < B", "A << B"):
        return "B"
    return "A"   # covers 'A > B', 'A >> B', or anything unexpected

def get_preference_strength(row: pd.Series) -> int:
    """
    Map final_choice strings to preference strength.
    '>' / '<'   → 1 (weak preference)
    '>>' / '<<' → 2 (strong preference)
    """
    choice = str(row.get("final_choice", "")).strip()
    if ">>" in choice or "<<" in choice:
        return 2
    if ">" in choice or "<" in choice:
        return 1
    return 1

def load_annotator_data(csv_path: str) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    return df

def sample_and_split(df: pd.DataFrame, seed: int) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Sample SAMPLE_SIZE rows, shuffle, split 20/20."""
    n = min(SAMPLE_SIZE, len(df))
    sampled = df.sample(n=n, random_state=seed).reset_index(drop=True)
    train = sampled.iloc[:TRAIN_SIZE].reset_index(drop=True)
    test  = sampled.iloc[TRAIN_SIZE:TRAIN_SIZE + EVAL_SIZE].reset_index(drop=True)
    return train, test


# ==============================================================================
# PHASE A — PAIR ANALYSIS (FEATURE EXTRACTION)
# ==============================================================================
def phase_a(train_df: pd.DataFrame, annotator_id: str) -> list:
    analysis_results = []
    print(f"\n  --- Phase A: Pair Analysis ({len(train_df)} pairs) ---")

    for idx, row in train_df.iterrows():
        pair_num = idx + 1
        cid = row.get("cid", "unknown")

        img_a_path = os.path.join(IMAGES_DIR, row["left_file"])
        img_b_path = os.path.join(IMAGES_DIR, row["right_file"])
        img_a_b64 = encode_image(img_a_path)
        img_b_b64 = encode_image(img_b_path)

        if not img_a_b64 or not img_b_b64:
            print(f"  -> Skipping (missing image)")
            continue

        winner = get_winner_label(row)
        preference_strength = get_preference_strength(row)
        final_choice_raw = str(row.get("final_choice", "")).strip()
        print(
            f"  Analyzing {pair_num}/{len(train_df)} "
            f"[cid={cid}] [final_choice={final_choice_raw}] "
            f"[winner={winner}] [strength={preference_strength}]..."
        )

        ANALYSIS_PROMPT = f"""You are an expert UX design researcher discovering a user's NICHE taste.
The user looked at Image A and Image B.
The user EXPLICITLY CHOSE: Image {winner}.
Look at the CHOSEN design (Image {winner}) and the REJECTED design.
Identify up to 5 visual dimensions where they differ (e.g., color_theme, spacing, corner_radius).
For each dimension, state exactly what visual trait the CHOSEN design has that the user apparently likes, and what the REJECTED design has.
Return your analysis as a JSON object with this strict schema:
{{
  "why_user_chose_winner": "<max 15 words summary of why they picked Image {winner}>",
  "preferred_features": [
    {{
      "dimension": "<snake_case_name>",
      "preferred_trait_in_winner": "<brief trait in Image {winner} (e.g., 'dark background')>",
      "rejected_trait_in_loser": "<brief trait in the other image (e.g., 'light background')>"
    }}
  ]
}}"""

        try:
            result = call_llm(ANALYSIS_PROMPT, img_a_b64, img_b_b64)
            llm_output_valid = is_valid_phase_a_result(result)
            analysis_results.append({
                "pair_id": row.get("pair_id"),
                "cid": cid,
                "human_choice": winner,
                "preference_strength": preference_strength,
                "final_choice_raw": final_choice_raw,
                "llm_output_valid": llm_output_valid,
                "llm_analysis": result,
            })
        except Exception as e:
            print(f"  -> Failed: {e}")
            analysis_results.append({
                "pair_id": row.get("pair_id"),
                "cid": cid,
                "human_choice": winner,
                "preference_strength": preference_strength,
                "final_choice_raw": final_choice_raw,
                "llm_output_valid": False,
                "llm_analysis": None,
                "error": str(e),
            })

    return analysis_results


# ==============================================================================
# PHASE C — SYNTHESIS & DYNAMIC CRITERIA EXTRACTION (WEIGHTED RUBRIC)
# ==============================================================================
def phase_c(analysis_results: list) -> tuple[str, list]:
    print(f"\n  --- Phase C: Synthesis ({len(analysis_results)} analyses) ---")

    observations = []
    for a in analysis_results:
        strength = int(a.get("preference_strength", 1) or 1)
        preferred_features = a.get("llm_analysis", {}).get("preferred_features", [])
        weighted_features = [
            {
                **feature,
                "evidence_weight": strength,
            }
            for feature in preferred_features
            if isinstance(feature, dict)
        ]
        observations.append({
            "final_choice_raw": a.get("final_choice_raw"),
            "preference_strength": strength,
            "preferred_features": weighted_features,
        })
    random.shuffle(observations)
    sample_observations = observations[:50]

    SYNTHESIS_PROMPT = f"""You are a UX researcher. You are given raw observations of what a specific user PREFERRED vs REJECTED across multiple UI pairs.
Raw Observations of Preferred vs Rejected traits:
{json.dumps(sample_observations, indent=2)}
Identify the strongest, most consistent patterns in what they PREFER.
Each observation includes `preference_strength` and each feature includes `evidence_weight`.
A weight of 1 means a weak preference (`<` or `>`). A weight of 2 means a strong preference (`<<` or `>>`).
CRITICAL INSTRUCTION:
1. Identify exactly the TOP 7 evaluation criteria based strictly on the WEIGHTED SUPPORT of similar visual dimensions across the raw observations.
2. Group similar concepts together, sum their weighted evidence, and assign a `weight` to each metric representing its weighted importance.
3. Strong-preference evidence (weight 2) must count more than weak-preference evidence (weight 1).
Return a JSON object with this schema:
{{
  "niche_preference_profile": "<Clear, exact description of the specific visual traits this user likes based on the data. Be very literal.>",
  "top_7_evaluation_criteria": [
    {{
      "criterion": "<snake_case_criterion_1>",
      "description": "<what trait the user prefers for this>",
      "weight": <numeric_float_based_on_weighted_support>
    }}
  ]
}}
"""

    print("  Synthesizing preference profile...")
    try:
        synthesis_result = call_llm(prompt=SYNTHESIS_PROMPT)
        niche_preference_profile = synthesis_result.get(
            "niche_preference_profile", "Could not synthesize profile."
        )
        top_7_criteria = synthesis_result.get("top_7_evaluation_criteria", [])
        if not top_7_criteria:
            raise ValueError("No criteria extracted")
    except Exception as e:
        print(f"  -> Synthesis failed: {e}")
        niche_preference_profile = "Synthesis failed."
        top_7_criteria = []

    # Clean & validate
    clean_top_7 = []
    for i, c in enumerate(top_7_criteria):
        if isinstance(c, dict):
            clean_top_7.append({
                "criterion": c.get("criterion", f"criterion_{i+1}"),
                "description": c.get("description", ""),
                "weight": float(c.get("weight", 1.0) if c.get("weight") is not None else 1.0),
            })
        else:
            clean_top_7.append({"criterion": f"criterion_{i+1}", "description": str(c), "weight": 1.0})

    # Fallback if empty
    if not clean_top_7:
        clean_top_7 = [
            {"criterion": f"fallback_{i+1}", "description": "fallback", "weight": 1.0}
            for i in range(7)
        ]

    print(f"  Profile: {niche_preference_profile[:100]}...")
    for c in clean_top_7:
        print(f"    - {c['criterion']} (weight={c['weight']})")

    return niche_preference_profile, clean_top_7


# ==============================================================================
# PHASE D — EVALUATION (SCORE × WEIGHT = OUTCOME)
# ==============================================================================
def phase_d(
    test_df: pd.DataFrame,
    niche_preference_profile: str,
    top_7_criteria: list,
) -> tuple[list, float, int, int]:
    print(f"\n  --- Phase D: Evaluation ({len(test_df)} pairs) ---")

    criteria_for_eval = [
        {"criterion": c["criterion"], "preferred_trait": c["description"]}
        for c in top_7_criteria
    ]
    criteria_json = json.dumps(criteria_for_eval, indent=2)

    EVAL_PROMPT = f"""You are an AI scoring interfaces for a user with this EXACT preference profile:
USER'S NICHE DESIGN PREFERENCE PROFILE:
"{niche_preference_profile}"
Criteria to evaluate:
{criteria_json}
CRITICAL CONSTRAINTS:
1. For EACH criterion, assign a numeric score from 1 to 10 for BOTH Image A and Image B indicating how well they match the user's preferred trait.
2. DO NOT use generic UI best practices. Use ONLY the user's specific profile.
Return a JSON object with this exact schema:
{{
  "criteria_evaluations": [
    {{
      "criterion": "<Must be from the 7 provided>",
      "score_a": <integer 1 to 10>,
      "score_b": <integer 1 to 10>,
      "reason": "<max 5 words>"
    }}
  ]
}}
"""

    weights_map = {c["criterion"]: float(c.get("weight", 1.0)) for c in top_7_criteria}
    eval_predictions = []
    correct_predictions = 0
    valid_evals = 0

    for idx, row in test_df.iterrows():
        pair_num = idx + 1
        cid = row.get("cid", "unknown")

        img_a_path = os.path.join(IMAGES_DIR, row["left_file"])
        img_b_path = os.path.join(IMAGES_DIR, row["right_file"])
        img_a_b64 = encode_image(img_a_path)
        img_b_b64 = encode_image(img_b_path)

        if not img_a_b64 or not img_b_b64:
            print(f"  -> Skipping (missing image)")
            continue

        true_winner = get_winner_label(row)
        final_choice_raw = str(row.get("final_choice", "")).strip()
        true_strength = get_preference_strength(row)
        print(
            f"  Evaluating {pair_num}/{len(test_df)} "
            f"[cid={cid}] [final_choice={final_choice_raw}] "
            f"[true_winner={true_winner}] [strength={true_strength}]..."
        )

        try:
            result = call_llm(EVAL_PROMPT, img_a_b64, img_b_b64)
            llm_output_valid = is_valid_phase_d_result(result)

            total_score_a = 0.0
            total_score_b = 0.0

            for eval_item in result.get("criteria_evaluations", []):
                crit_name = eval_item.get("criterion", "")
                try:
                    score_a = float(eval_item.get("score_a", 0))
                except (ValueError, TypeError):
                    score_a = 0.0
                try:
                    score_b = float(eval_item.get("score_b", 0))
                except (ValueError, TypeError):
                    score_b = 0.0

                weight = weights_map.get(crit_name, 1.0)
                total_score_a += score_a * weight
                total_score_b += score_b * weight

                eval_item["calculated_weighted_score_a"] = score_a * weight
                eval_item["calculated_weighted_score_b"] = score_b * weight
                eval_item["applied_weight"] = weight

            predicted_choice = "A" if total_score_a >= total_score_b else "B"
            result["final_total_score_a"] = total_score_a
            result["final_total_score_b"] = total_score_b
            result["derived_predicted_choice"] = predicted_choice

            is_correct = predicted_choice == true_winner
            if is_correct:
                correct_predictions += 1
            valid_evals += 1

            eval_predictions.append({
                "pair_id": row.get("pair_id"),
                "cid": cid,
                "true_winner": true_winner,
                "final_choice_raw": final_choice_raw,
                "true_strength": true_strength,
                "predicted_choice": predicted_choice,
                "is_correct": is_correct,
                "llm_output_valid": llm_output_valid,
                "details": result,
            })

            running_accuracy = (correct_predictions / valid_evals) if valid_evals > 0 else 0.0
            verdict = "RIGHT" if is_correct else "WRONG"
            print(
                f"  -> A={total_score_a:.2f} | B={total_score_b:.2f} | "
                f"Pred={predicted_choice} | True={true_winner} | {verdict} | "
                f"Parsed={llm_output_valid} | RunningAcc={correct_predictions}/{valid_evals} ({running_accuracy*100:.2f}%)"
            )

        except Exception as e:
            print(f"  -> Failed: {e}")
            eval_predictions.append({
                "pair_id": row.get("pair_id"),
                "cid": cid,
                "true_winner": true_winner,
                "final_choice_raw": final_choice_raw,
                "true_strength": true_strength,
                "predicted_choice": None,
                "is_correct": False,
                "llm_output_valid": False,
                "details": None,
                "error": str(e),
            })

    accuracy = (correct_predictions / valid_evals) if valid_evals > 0 else 0.0
    print(f"  Accuracy: {correct_predictions}/{valid_evals} ({accuracy*100:.2f}%)")
    return eval_predictions, accuracy, correct_predictions, valid_evals


# ==============================================================================
# MAIN — ITERATE OVER ALL ANNOTATORS
# ==============================================================================
def run_pipeline_for_annotator(csv_path: str, annotator_id: str):
    print(f"\n{'='*60}")
    print(f"Processing annotator: {annotator_id}")
    print(f"{'='*60}")

    set_seed(SEED)

    df = load_annotator_data(csv_path)
    print(f"  Loaded {len(df)} rows from {csv_path}")

    if len(df) < SAMPLE_SIZE:
        print(f"  Warning: only {len(df)} rows available, sampling all.")

    train_df, test_df = sample_and_split(df, seed=SEED)
    print(f"  Train: {len(train_df)} | Test: {len(test_df)}")

    # Phase A
    analysis_results = phase_a(train_df, annotator_id)

    # Phase C
    niche_preference_profile, top_7_criteria = phase_c(analysis_results)

    # Phase D
    eval_predictions, accuracy, correct, total_eval = phase_d(
        test_df, niche_preference_profile, top_7_criteria
    )

    # Build output
    output = {
        "annotator_id": annotator_id,
        "model": MODEL,
        "sample_sizes": {
            "total_sampled": len(train_df) + len(test_df),
            "train": len(train_df),
            "test": len(test_df),
        },
        "niche_preference_profile": niche_preference_profile,
        "top_7_criteria": top_7_criteria,
        "pair_analyses": analysis_results,
        "evaluation_metrics": {
            "accuracy": accuracy,
            "correct": correct,
            "total_evaluated": total_eval,
        },
        "predictions": eval_predictions,
    }

    # Save
    os.makedirs(RESULTS_DIR, exist_ok=True)
    out_path = os.path.join(RESULTS_DIR, f"{annotator_id}.json")
    with open(out_path, "w") as f:
        json.dump(output, f, indent=2)
    print(f"\n  Saved → {out_path}")
    return output


def main():
    annotator_dir = Path(ANNOTATOR_DIR)
    if not annotator_dir.exists():
        print(f"Error: {ANNOTATOR_DIR} directory not found.")
        return

    csv_files = sorted(annotator_dir.glob("*.csv"))
    if not csv_files:
        print(f"No CSV files found in {ANNOTATOR_DIR}")
        return

    print(f"Found {len(csv_files)} annotator(s): {[f.stem for f in csv_files]}")

    for csv_path in csv_files:
        annotator_id = csv_path.stem
        try:
            run_pipeline_for_annotator(str(csv_path), annotator_id)
        except Exception as e:
            print(f"\n[ERROR] Failed for annotator {annotator_id}: {e}")

    print("\nAll annotators processed.")


if __name__ == "__main__":
    main()

Found 20 annotator(s): ['0c65ba0b46894372', '1d3ee9b46ac34e6c', '247b7dfa8a5347ad', '2c79548f2ab44243', '2d8219ac74d146ba', '454c297fa1e54296', '498b9ea72d994e8e', '653f05d5c9ab4c97', '6ccbe484cb96425f', '7602a5d37b7d4220', '7a945bb87f2b4cd2', '8287fcc5b6504e39', '8f61f5a0685f42ec', '97945b44a2b54914', 'a692cdb11bbe432c', 'abda3f52c24c4038', 'ad7def63e86045b1', 'dad4b876ad3147e4', 'e8f37a526c444958', 'eee1fd24aad648ed']

Processing annotator: 0c65ba0b46894372
  Loaded 610 rows from annotator/0c65ba0b46894372.csv
  Train: 20 | Test: 20

  --- Phase A: Pair Analysis (20 pairs) ---
  Analyzing 1/20 [cid=10681::gpt_comp_s2w_10681_v1.png>>gpt_comp_s2w_10681_v2.png::T000000] [final_choice=A >> B] [winner=A] [strength=2]...
  Analyzing 2/20 [cid=10681::gpt_comp_s2w_10681_v1.png>>gpt_comp_s2w_10681_v3.png::T000001] [final_choice=A < B] [winner=B] [strength=1]...
  Analyzing 3/20 [cid=10681::gpt_comp_s2w_10681_v1.png>>gpt_comp_s2w_10681_v4.png::T000002] [final_choice=A < B] [winner=B] [strength

KeyboardInterrupt: 